In [3]:
# 01_data_clean.ipynb - 修复下载函数（改用requests库，兼容所有Python版本）
import pandas as pd
import numpy as np
import re
import requests  # 替换urllib，解决timeout兼容问题
from pathlib import Path
import time

# 1. 先检查并安装requests库（若未安装）
def install_requests():
    try:
        import requests
        print("✅ requests库已安装，可正常使用")
    except ImportError:
        print("⚠️ requests库未安装，正在自动安装...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
        print("✅ requests库安装完成")

# 执行安装检查
install_requests()
import requests  # 确保导入成功

# 2. 路径定义（同级目录结构）
DATA_RAW = Path("data_raw")       # 同级的data_raw文件夹
DATA_CLEAN = Path("data_clean")   # 同级的data_clean文件夹

# 确保基础文件夹存在
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_CLEAN.mkdir(parents=True, exist_ok=True)
print(f"\n✅ 基础文件夹已准备：\n- {DATA_RAW.absolute()}\n- {DATA_CLEAN.absolute()}")

# 3. 修复：使用requests库自动下载数据（兼容所有Python版本）
def auto_download_data():
    """自动下载Mfytop15（进口）和xfytop15（出口）数据（改用requests库）"""
    # 官方下载地址（你提供的链接）
    data_urls = {
        "Mfytop15.xlsx": "https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/50441/Mfytop15.xlsx?v=26152",  # 进口
        "xfytop15.xlsx": "https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/50441/xfytop15.xlsx?v=60078"   # 出口
    }
    
    downloaded_files = []
    for file_name, url in data_urls.items():
        file_path = DATA_RAW / file_name
        
        # 检查文件是否已存在，避免重复下载
        if file_path.exists() and file_path.stat().st_size > 0:
            print(f"ℹ️ {file_name} 已存在且有效，跳过下载")
            downloaded_files.append(file_path)
            continue
        
        # 开始下载（使用requests库，设置超时10秒）
        print(f"\n📥 正在下载 {file_name}...")
        retry_count = 2  # 重试2次
        success = False
        
        for i in range(retry_count):
            try:
                # 发送请求下载文件（设置超时10秒，添加用户代理模拟浏览器）
                headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
                response = requests.get(url, headers=headers, timeout=10, stream=True)
                response.raise_for_status()  # 若状态码非200，抛出异常
                
                # 保存文件到data_raw
                with open(file_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024*1024):  # 分块下载（1MB/块）
                        if chunk:
                            f.write(chunk)
                
                # 验证文件是否有效（大小>0）
                if file_path.stat().st_size > 0:
                    print(f"✅ {file_name} 下载成功（第{i+1}次尝试），保存路径：{file_path.absolute()}")
                    success = True
                    downloaded_files.append(file_path)
                    break
                else:
                    print(f"⚠️ {file_name} 下载后文件为空（第{i+1}次尝试）")
            
            except Exception as e:
                error_msg = str(e)[:100]  # 截取前100字符，避免日志过长
                if i < retry_count - 1:
                    print(f"⚠️ {file_name} 下载失败（第{i+1}次尝试）：{error_msg}，2秒后重试...")
                    time.sleep(2)
                else:
                    print(f"❌ {file_name} 多次下载失败：{error_msg}")
        
        if not success:
            raise FileNotFoundError(
                f"❌ {file_name} 下载失败，请手动下载后放入以下文件夹：\n"
                f"目标路径：{DATA_RAW.absolute()}\n"
                f"手动下载地址：{url}"
            )
    
    # 确认两个文件都已下载成功
    if len(downloaded_files) != 2:
        missing = [name for name in data_urls.keys() if not (DATA_RAW / name).exists()]
        raise FileNotFoundError(f"❌ 缺失以下文件：{', '.join(missing)}，请手动补充")
    
    return DATA_RAW / "Mfytop15.xlsx", DATA_RAW / "xfytop15.xlsx"

# 4. 读取并清洗数据（适配表格格式，无修改）
def read_and_clean_data(file_path, trade_type):
    """读取数据并完成宽表转长表"""
    # 从第三行读取表头（header=2，匹配表格格式）
    df_raw = pd.read_excel(file_path, header=2)
    print(f"\n🔍 {trade_type}数据 - 原始形状: {df_raw.shape}")
    
    # 清理空列（删除全为空的列）
    df_clean_cols = df_raw.dropna(axis=1, how="all")
    print(f"🔍 清理空列后形状: {df_clean_cols.shape}")
    
    # 识别年份组（Country列 + Fiscal year XXXX列）
    country_cols = [col for col in df_clean_cols.columns if "Country" in str(col)]
    year_groups = []
    
    for country_col in country_cols:
        col_idx = df_clean_cols.columns.get_loc(country_col)
        # 确保金额列存在（不超出列范围）
        if col_idx + 1 < len(df_clean_cols.columns):
            amount_col = df_clean_cols.columns[col_idx + 1]
            # 从金额列名提取年份（如"Fiscal year 2024" → 2024）
            year_match = re.search(r"(\d{4})", str(amount_col))
            if year_match and "Fiscal year" in str(amount_col):
                year_groups.append((int(year_match.group(1)), country_col, amount_col))
    
    if len(year_groups) == 0:
        raise ValueError(f"❌ 未识别到年份组，请检查{file_path.name}的表格格式")
    
    print(f"✅ 识别到{len(year_groups)}个年份组（示例：{year_groups[:3]}）")
    
    # 提取每个年份组的有效数据
    all_year_data = []
    for year, country_col, amount_col in year_groups:
        # 提取当前年份的国家和金额
        year_data = df_clean_cols[[country_col, amount_col]].copy()
        
        # 清洗国家列（去除空格、空字符串）
        year_data[country_col] = year_data[country_col].astype(str).str.strip()
        year_data = year_data[
            (year_data[country_col] != "") & 
            (year_data[country_col] != "nan") & 
            (year_data[country_col].notna())
        ]
        
        # 清洗金额列（转为数值，排除0和空）
        year_data[amount_col] = pd.to_numeric(year_data[amount_col], errors="coerce")
        year_data = year_data.dropna(subset=[amount_col])
        year_data = year_data[year_data[amount_col] > 0]
        
        # 统一列名并添加标签
        year_data.rename(columns={
            country_col: "Country",
            amount_col: "value_usd"
        }, inplace=True)
        year_data["fiscal_year"] = year
        year_data["trade_type"] = trade_type
        
        # 保留Top15数据（符合文件命名逻辑）
        all_year_data.append(year_data.head(15))
    
    # 合并所有年份数据
    final_df = pd.concat(all_year_data, ignore_index=True)
    print(f"✅ {trade_type}数据处理完成 - 总行数: {len(final_df)}")
    return final_df

# 5. 保存清洗后的数据（无修改）
def save_clean_data(import_df, export_df):
    """保存清洗后的长格式数据"""
    # 保存进口数据
    import_path = DATA_CLEAN / "us_import_long.csv"
    import_df.to_csv(import_path, index=False)
    # 保存出口数据
    export_path = DATA_CLEAN / "us_export_long.csv"
    export_df.to_csv(export_path, index=False)
    # 保存合并数据
    combined_path = DATA_CLEAN / "us_trade_combined.csv"
    combined_df = pd.concat([import_df, export_df], ignore_index=True)
    combined_df.to_csv(combined_path, index=False)
    
    print(f"\n💾 清洗数据保存完成：")
    print(f"- 进口数据: {import_path.absolute()}（{len(import_df)}行）")
    print(f"- 出口数据: {export_path.absolute()}（{len(export_df)}行）")
    print(f"- 合并数据: {combined_path.absolute()}（{len(combined_df)}行）")

# 6. 主执行流程（自动下载→读取→清洗→保存）
if __name__ == "__main__":
    try:
        # 步骤1：自动下载原始数据
        print("=== 步骤1：自动下载数据 ===")
        import_file, export_file = auto_download_data()
        
        # 步骤2：读取并清洗进口数据
        print("\n=== 步骤2：处理进口数据（Mfytop15.xlsx）===")
        us_import_df = read_and_clean_data(import_file, "import")
        
        # 步骤3：读取并清洗出口数据
        print("\n=== 步骤3：处理出口数据（xfytop15.xlsx）===")
        us_export_df = read_and_clean_data(export_file, "export")
        
        # 步骤4：保存清洗后的数据
        print("\n=== 步骤4：保存清洗数据 ===")
        save_clean_data(us_import_df, us_export_df)
        
        print(f"\n🎉 所有数据清洗流程完成！可继续运行02_data_analysis.ipynb进行分析")
        
    except Exception as e:
        print(f"\n❌ 流程执行失败：{str(e)}")
        # 简化错误日志，避免冗余
        print("💡 若自动下载持续失败，建议手动下载：")
        print("1. 进口数据：https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/50441/Mfytop15.xlsx?v=26152")
        print("2. 出口数据：https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/50441/xfytop15.xlsx?v=60078")
        print(f"3. 手动放入文件夹：{DATA_RAW.absolute()}")

✅ requests库已安装，可正常使用

✅ 基础文件夹已准备：
- c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\data_raw
- c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\data_clean
=== 步骤1：自动下载数据 ===

📥 正在下载 Mfytop15.xlsx...
✅ Mfytop15.xlsx 下载成功（第1次尝试），保存路径：c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\data_raw\Mfytop15.xlsx

📥 正在下载 xfytop15.xlsx...
✅ xfytop15.xlsx 下载成功（第1次尝试），保存路径：c:\Users\wujunyi\Desktop\小谢旁听\2026春-数据分析概论\homework\ex_team6-group\data_raw\xfytop15.xlsx

=== 步骤2：处理进口数据（Mfytop15.xlsx）===

🔍 import数据 - 原始形状: (18, 104)
🔍 清理空列后形状: (18, 70)
✅ 识别到35个年份组（示例：[(2024, 'Country ', 'Fiscal year 2024'), (2023, 'Country .1', 'Fiscal year 2023'), (2022, 'Country .2', 'Fiscal year 2022')]）
✅ import数据处理完成 - 总行数: 525

=== 步骤3：处理出口数据（xfytop15.xlsx）===

🔍 export数据 - 原始形状: (18, 104)
🔍 清理空列后形状: (18, 70)
✅ 识别到35个年份组（示例：[(2024, 'Country ', 'Fiscal year 2024'), (2023, 'Country .1', 'Fiscal year 2023'), (2022, 'Country .2', 'Fiscal year 2022')]）
✅ export数据处理完成 - 